# gf2^8_mult timing plots

`summary.csv` から `gf2^8_mult.tfc` の結果を読み込み、以下の2種類の図を出力します。

- 横軸: 枝刈りパラメータ `p`、縦軸: 総実行時間
- 枝刈り時間の内訳の積み上げ棒グラフ


In [3]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Notebook の場所に依存せず、TOpt リポジトリ直下を探します。
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "forme" / "sketch_experiments").exists():
            return path
    raise FileNotFoundError("Could not find TOpt repository root")

ROOT = find_repo_root()
SUMMARY_CSV = ROOT / "forme" / "sketch_experiments" / "aa_sketch_detail_20260618_032309" / "summary.csv"
OUT_DIR = ROOT / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CIRCUIT = "gf2^8_mult.tfc"

print(ROOT)
print(SUMMARY_CSV)

In [ ]:
def as_float(row, key, default=0.0):
    value = row.get(key, "")
    return default if value == "" else float(value)


def as_int(row, key):
    value = row.get(key, "")
    return None if value == "" else int(float(value))


with SUMMARY_CSV.open(newline="") as f:
    all_rows = list(csv.DictReader(f))

rows = [
    row for row in all_rows
    if row.get("circuit") == CIRCUIT and row.get("variant") in {"reuse", "noreuse"}
]
rows.sort(key=lambda row: (as_int(row, "p"), row["variant"]))

baseline = next(
    (row for row in all_rows if row.get("circuit") == CIRCUIT and row.get("variant") == "baseline"),
    None,
)

def set_zero_based_ylim(ax, max_value, pad=0.08):
    upper = max_value * (1.0 + pad) if max_value > 0 else 1.0
    ax.set_ylim(bottom=0, top=upper)


def common_p_values(by_variant, variants):
    p_sets = [{as_int(row, "p") for row in by_variant[variant]} for variant in variants]
    common = sorted(set.intersection(*p_sets))
    if not common:
        raise ValueError(f"No common p values found for variants: {', '.join(variants)}")
    return common

print(f"rows: {len(rows)}")
print(f"p values: {sorted({as_int(row, 'p') for row in rows})}")
print(f"baseline total_exec_s: {as_float(baseline, 'total_exec_s') if baseline else 'none'}")

In [ ]:
# 確認用の小さな表です。
preview_columns = [
    "variant", "p", "total_exec_s", "sketch_ms", "sketch_build_ms",
    "sketch_basis_ms", "sketch_mem_ms", "sketch_other_ms",
]

for row in rows:
    print({key: row.get(key, "") for key in preview_columns})

## 1. 総実行時間

横軸を枝刈りパラメータ `p`、縦軸を `total_exec_s` として、基底再利用あり/なしを比較します。

In [ ]:
by_variant = {"reuse": [], "noreuse": []}
for row in rows:
    by_variant[row["variant"]].append(row)

fig, ax = plt.subplots(figsize=(7.2, 4.4), constrained_layout=True)
max_time = 0.0

styles = {
    "reuse": {
        "label": "with basis reuse",
        "marker": "o",
        "linewidth": 2.2,
        "color": "#1f77b4",
    },
    "noreuse": {
        "label": "without basis reuse",
        "marker": "s",
        "linewidth": 2.2,
        "linestyle": "--",
        "color": "#d95f02",
    },
}

for variant in ["reuse", "noreuse"]:
    data = by_variant[variant]
    if not data:
        raise ValueError(f"No rows found for variant={variant!r}")
    ps = [as_int(row, "p") for row in data]
    times = [as_float(row, "total_exec_s") for row in data]
    max_time = max(max_time, max(times))
    ax.plot(ps, times, **styles[variant])

if baseline:
    baseline_time = as_float(baseline, "total_exec_s")
    max_time = max(max_time, baseline_time)
    ax.axhline(
        baseline_time,
        color="0.35",
        linewidth=1.5,
        linestyle=":",
        label=f"baseline ({baseline_time:.1f} s)",
    )

ax.set_title("gf2^8_mult: total execution time")
ax.set_xlabel("pruning parameter p")
ax.set_ylabel("total execution time [s]")
set_zero_based_ylim(ax, max_time)
ax.legend(frameon=False)

fig.savefig(OUT_DIR / "gf2_8_mult_timing_total_time.png")
fig.savefig(OUT_DIR / "gf2_8_mult_timing_total_time.pdf")
plt.show()

## 2. 枝刈り時間の内訳

`sketch_ms` の内訳として、以下を積み上げ棒グラフで表示します。

- `sketch_build_ms`
- `sketch_basis_ms`
- `sketch_mem_ms`
- `sketch_other_ms`


In [ ]:
# "both", "reuse", "noreuse" のどれかを指定できます。
BREAKDOWN_VARIANT = "both"

In [ ]:
components = [
    ("sketch_build_ms", "build sketch"),
    ("sketch_basis_ms", "basis work"),
    ("sketch_mem_ms", "memory check"),
    ("sketch_other_ms", "other sketch"),
]
colors = ["#4C78A8", "#72B7B2", "#F58518", "#B279A2"]

variants = ["reuse", "noreuse"] if BREAKDOWN_VARIANT == "both" else [BREAKDOWN_VARIANT]
ps = common_p_values(by_variant, variants)

fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)

if len(variants) == 1:
    width = 0.62
    x_positions = {p: i for i, p in enumerate(ps)}
    tick_positions = list(x_positions.values())
    tick_labels = [str(p) for p in ps]
else:
    width = 0.36
    x_positions = {}
    tick_positions = []
    tick_labels = []
    for i, p in enumerate(ps):
        x_positions[(p, "reuse")] = i - width / 2
        x_positions[(p, "noreuse")] = i + width / 2
        tick_positions.append(i)
        tick_labels.append(str(p))

for variant in variants:
    data_by_p = {as_int(row, "p"): row for row in by_variant[variant]}
    bottoms = [0.0] * len(ps)
    for (column, label), color in zip(components, colors):
        values = [as_float(data_by_p[p], column) / 1000.0 for p in ps]
        if len(variants) == 1:
            xs = [x_positions[p] for p in ps]
            legend_label = label
        else:
            xs = [x_positions[(p, variant)] for p in ps]
            legend_label = label if variant == variants[0] else None
        ax.bar(xs, values, width, bottom=bottoms, label=legend_label, color=color)
        bottoms = [bottom + value for bottom, value in zip(bottoms, values)]

max_stacked_time = max(
    sum(as_float(row, column) / 1000.0 for column, _ in components)
    for variant in variants
    for row in by_variant[variant]
)

if len(variants) == 2:
    for p in ps:
        ax.text(x_positions[(p, "reuse")], -0.06, "R", ha="center", va="top", transform=ax.get_xaxis_transform(), fontsize=9)
        ax.text(x_positions[(p, "noreuse")], -0.06, "N", ha="center", va="top", transform=ax.get_xaxis_transform(), fontsize=9)
    subtitle = "R: with reuse, N: without reuse"
else:
    subtitle = "with basis reuse" if BREAKDOWN_VARIANT == "reuse" else "without basis reuse"

ax.set_title(f"gf2^8_mult: pruning-time breakdown ({subtitle})")
ax.set_xlabel("pruning parameter p")
ax.set_ylabel("pruning time [s]")
ax.set_xticks(tick_positions, tick_labels)
set_zero_based_ylim(ax, max_stacked_time)
ax.legend(frameon=False, ncol=2)

suffix = "pruning_breakdown" if BREAKDOWN_VARIANT == "both" else f"pruning_breakdown_{BREAKDOWN_VARIANT}"
fig.savefig(OUT_DIR / f"gf2_8_mult_timing_{suffix}.png")
fig.savefig(OUT_DIR / f"gf2_8_mult_timing_{suffix}.pdf")
plt.show()